# Sequential Architecture — CrewAI

**Pattern:** Sequential (A → B → C pipeline)  
**Framework:** CrewAI  
**Task:** Research city data → Summarize → Format travel report

## Crew Structure

```
┌──────────────────────────────────────────────────────┐
│                       CREW                           │
│  Process: Sequential                                 │
│                                                      │
│  ┌────────────┐  ┌────────────┐  ┌────────────────┐  │
│  │ Researcher │  │ Summarizer │  │   Formatter    │  │
│  │ tools: 1   │  │ tools: []  │  │   tools: []    │  │
│  └─────┬──────┘  └─────┬──────┘  └────────────────┘  │
│        │               │                  ▲           │
│  ┌─────▼──────┐  ┌─────▼──────┐  ┌────────┴───────┐  │
│  │  Task 1:   │  │  Task 2:   │  │    Task 3:     │  │
│  │  Research  │─▶│  Summarize │─▶│    Format      │  │
│  └────────────┘  └────────────┘  │ output_pydantic│  │
│  context=[]      context=[t1]    │ context=[t2]   │  │
│                                  └────────────────┘  │
└──────────────────────────────────────────────────────┘
```

**CrewAI difference from LangChain/LangGraph:**  
`context=[prev_task]` passes the previous task's output explicitly. Agents have roles and backstories — they're closer to simulated team members than pure function chains.

## Setup

In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"

gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)
print("✓ Ready")

## Output Schema + Mock Tool

In [ ]:
class TravelReport(BaseModel):
    city: str = Field(description="City name")
    report_section: str = Field(description="Full formatted report section with markdown header")
    summary: str = Field(description="One-sentence traveler summary")

@tool("City Data Fetcher")
def fetch_city_data(city: str) -> str:
    """Fetch raw travel data for a city. Input: city name."""
    data = {
        "tokyo":     "Weather: Clear, 18°C. Safety: Low. Time: 22:30 JST. Highlights: Shibuya, Senso-ji.",
        "paris":     "Weather: Partly Cloudy, 16°C. Safety: Low. Time: 15:30 CET. Highlights: Eiffel Tower, Louvre.",
        "bangalore": "Weather: Rainy, 25°C. Safety: Medium. Time: 20:00 IST. Highlights: Lalbagh, Nandi Hills.",
    }
    return data.get(city.lower(), f"No data for '{city}'.")

print("Schema fields:", list(TravelReport.model_fields.keys()))
print("Tool test:", fetch_city_data.run("Tokyo"))

## Three Agents with Role Separation

In [ ]:
researcher = Agent(
    role="Travel Data Researcher",
    goal="Fetch and structure raw travel data for the given city.",
    backstory="You gather factual travel data — weather, safety, time, attractions — from data sources.",
    tools=[fetch_city_data],
    llm=gemini,
    verbose=False,
)

summarizer = Agent(
    role="Travel Summarizer",
    goal="Write a concise 2-sentence traveler summary from structured research data.",
    backstory="You distill dense travel data into clear, engaging 2-sentence summaries.",
    tools=[],
    llm=gemini,
    verbose=False,
)

formatter = Agent(
    role="Report Formatter",
    goal="Format the travel summary into a polished markdown report section.",
    backstory="You transform travel summaries into professional report sections with proper headers.",
    tools=[],
    llm=gemini,
    verbose=False,
)

print(f"Agents: {researcher.role} | {summarizer.role} | {formatter.role}")

## Tasks with `context=` Chain

In [ ]:
def build_crew(city: str) -> Crew:
    research_task = Task(
        description=f"Fetch and structure travel data for {city}: weather, safety, local time, top attractions.",
        expected_output=f"Structured travel facts for {city}.",
        agent=researcher,
    )
    summarize_task = Task(
        description=f"Write a 2-sentence traveler summary for {city} using the research data.",
        expected_output="A 2-sentence traveler summary.",
        agent=summarizer,
        context=[research_task],          # ← passes researcher output forward
    )
    format_task = Task(
        description=f"Format the summary for {city} into a markdown report section with '### {city}' header.",
        expected_output="A complete TravelReport with city, report_section, and summary.",
        agent=formatter,
        context=[summarize_task],         # ← passes summarizer output forward
        output_pydantic=TravelReport,
    )
    return Crew(
        agents=[researcher, summarizer, formatter],
        tasks=[research_task, summarize_task, format_task],
        process=Process.sequential,
        verbose=False,
    )

print("Crew factory ready")

## Run

In [ ]:
cities = ["Tokyo", "Paris", "Bangalore"]
sections = []

for city in cities:
    print(f"\nProcessing: {city}")
    result = build_crew(city).kickoff()
    report: TravelReport = result.pydantic
    if report:
        sections.append(report.report_section)
        print(f"  Summary: {report.summary}")
    else:
        sections.append(result.raw)

print("\n# Travel Report\n")
for s in sections:
    print(s)
    print()

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `context=[prev_task]` | Explicit data passing — CrewAI injects previous task's output |
| Role separation (tools only on Researcher) | Not all agents need tools — mirrors real team structure |
| `output_pydantic=TravelReport` | Final output is a validated Python object |
| `Process.sequential` | Enforces task order at the framework level |

### CrewAI vs LangChain for Sequential
| | LangChain | CrewAI |
|---|---|---|
| Data passing | Python variables | `context=[task]` |
| Step identity | Chains (functions) | Agents with roles |
| Output typing | `StrOutputParser` | `output_pydantic=` |
| Verbosity | Verbose logs | Task/agent descriptions |

### Next: [02-Parallel →](../../02-Parallel/CrewAI/parallel.ipynb)